# Fine-tuning CamemBERT
Model: camembert-base  
Task: Binary sentiment classification (Positive / Negative)  
Dataset: Allociné (French movie reviews)  
Target: Beat TF-IDF baseline F1 > 94%


Fine-tuning means taking a pre-trained model (CamemBERT) that already understands French language, and adapting it specifically for our sentiment classification task. Instead of training from scratch, we only adjust the model's weights slightly on our dataset — this is much faster and gives better results.

## 1. Setup & Dependencies

We install all required libraries. On Google Colab, we need to install them at the start of each session.

In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (CamembertTokenizer, CamembertForSequenceClassification,
    TrainingArguments,Trainer)
import evaluate

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## 2. Load & Tokenize Dataset

We load the Allociné dataset and tokenize it using the CamemBERT tokenizer. Each review is converted into token IDs that the model can process. We use MAX_LENGTH=256 as determined during preprocessing — it covers 100% of reviews in our dataset.

In [4]:
MODEL_NAME = 'camembert-base'
MAX_LENGTH = 256

tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)
dataset   = load_dataset('allocine')

def tokenize(batch):
    return tokenizer(batch['review'],padding='max_length',truncation=True,
        max_length=MAX_LENGTH )

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

train_dataset = tokenized['train']
val_dataset   = tokenized['validation']
test_dataset  = tokenized['test']

print(f'Train : {len(train_dataset)}')
print(f'Val   : {len(val_dataset)}')
print(f'Test  : {len(test_dataset)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Train : 160000
Val   : 20000
Test  : 20000


## 3. Load Pre-trained CamemBERT Model

We load `camembert-base` with a classification head on top. The model was pre-trained on a large French text corpus and already understands French grammar, vocabulary, and context. We just add a final layer that outputs 2 values: probability of Negative and Positive sentiment.

In [5]:
model = CamembertForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2,
    id2label={0: 'Negative', 1: 'Positive'},label2id={'Negative': 0, 'Positive': 1})

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters : {model.num_parameters():,}')

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: camembert-base
Parameters : 110,623,490


Model loaded: camembert-base

110,623,490 parameters: this is a large model compared to TF-IDF

UNEXPECTED keys: not a problem
These are weights from the pre-training task (language modeling) that are no longer needed for classification. They are automatically ignored.

MISSING keys: expected and normal
These are the new classification layers (classifier.dense, classifier.out_proj) that didn't exist in the original model. They are randomly initialized and will be learned during fine-tuning.

In short: The model loaded correctly. The warnings just mean CamemBERT is being adapted from a language model to a sentiment classifier — exactly what fine-tuning is supposed to do.


## 4. Define Metrics

We define the evaluation metrics used during training. After each epoch, the Trainer evaluates the model on the validation set and computes accuracy and F1-score. This allows us to monitor progress and detect overfitting early.

In [6]:
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1       = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    return {'accuracy': accuracy['accuracy'],'f1': f1['f1']}

## 5. Training Configuration

We define the training hyperparameters:
- **learning_rate**: how fast the model updates its weights — too high = unstable, too low = slow
- **num_train_epochs**: number of times the model sees the full training set
- **batch_size**: number of reviews processed at once — limited by GPU memory
- **warmup_steps**: gradually increases learning rate at the start to stabilize training
- **weight_decay**: regularization to prevent overfitting
- **load_best_model_at_end**: automatically keeps the best checkpoint based on F1

In [8]:
training_args = TrainingArguments(
    output_dir='./results/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./results/logs',
    logging_steps=100,
    report_to='none',
    fp16=torch.cuda.is_available()
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 6. Train the Model

We use the HuggingFace `Trainer` which handles the full training loop automatically — forward pass, loss computation, backpropagation, and evaluation. Training 3 epochs on 160k reviews takes approximately 30-45 minutes on a T4 GPU (Google Colab).

In [9]:
trainer = Trainer( model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset,compute_metrics=compute_metrics)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.121157,0.115415,0.966400,0.966389
2,0.085430,0.109319,0.970900,0.970902
3,0.050641,0.134418,0.971300,0.971303


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=30000, training_loss=0.0918951758146286, metrics={'train_runtime': 7145.299, 'train_samples_per_second': 67.177, 'train_steps_per_second': 4.199, 'total_flos': 6.31466532864e+16, 'train_loss': 0.0918951758146286, 'epoch': 3.0})


- Train loss decreases consistently each epoch → the model is learning well
- Val loss increases slightly at epoch 3 → minor overfitting starting
- F1 improves each epoch → best checkpoint saved at epoch 3
- The model converged well within 3 epochs

## 7. Final Evaluation on Test Set

After training, we evaluate the best model checkpoint on the test set — the data the model has never seen. This is the official score we compare against the TF-IDF baseline (F1 = 94.06%).

In [10]:
results = trainer.evaluate(test_dataset)

print('=== TEST RESULTS ===')
print(f"Accuracy : {results['eval_accuracy']:.4f} ({results['eval_accuracy']*100:.2f}%)")
print(f"F1-score : {results['eval_f1']:.4f} ({results['eval_f1']*100:.2f}%)")
print()
print('=== COMPARISON WITH BASELINE ===')
print(f"TF-IDF Baseline F1 : 94.06%")
print(f"CamemBERT F1       : {results['eval_f1']*100:.2f}%")
improvement = (results['eval_f1'] - 0.9406) * 100
print(f"Improvement        : {improvement:+.2f}%")

=== TEST RESULTS ===
Accuracy : 0.9718 (97.17%)
F1-score : 0.9718 (97.18%)

=== COMPARISON WITH BASELINE ===
TF-IDF Baseline F1 : 94.06%
CamemBERT F1       : 97.18%
Improvement        : +3.12%


CamemBERT clearly beats the baseline — the additional complexity is fully justified.

## 8. Save the Model

We save the fine-tuned model and tokenizer so we can reload them later for evaluation or inference — without retraining.

In [11]:
import os
import pickle

os.makedirs('./results/models/camembert-sentiment', exist_ok=True)

# Save model and tokenizer
trainer.save_model('./results/models/camembert-sentiment')
tokenizer.save_pretrained('./results/models/camembert-sentiment')

# Save metrics
os.makedirs('./results/metrics', exist_ok=True)
camembert_metrics = {
    'model':         'camembert-base fine-tuned',
    'test_accuracy': round(results['eval_accuracy'], 4),
    'test_f1':       round(results['eval_f1'], 4),
}
pickle.dump(camembert_metrics, open('./results/metrics/camembert_metrics.pkl', 'wb'))

print('Model and metrics saved!')
print(camembert_metrics)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and metrics saved!
{'model': 'camembert-base fine-tuned', 'test_accuracy': 0.9718, 'test_f1': 0.9718}


## 9. Interpretation

**Results obtained:**
- Test Accuracy : 97.18%
- Test F1-score : 97.18%

**Comparison with baseline:**
- TF-IDF baseline F1 : 94.06%
- CamemBERT F1       : 97.18%
- Improvement        : +3.12%

**What this means:**
- CamemBERT clearly beats the TF-IDF baseline by +3.12%
- The improvement comes from CamemBERT's ability to understand
  context, negation and irony that TF-IDF cannot capture
- Best epoch: 2 (lowest validation loss)
- Training time: ~2 hours on T4 GPU (Google Colab)